In [0]:
from pyspark.sql import functions as F         
# 1. Criar o banco de dados Gold
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

# 2. Carregar as tabelas da Silver que usaremos nos projetos
df_pedidos = spark.table("silver.fat_pedido_total")
df_itens = spark.table("silver.fat_itens_pedidos")
df_produtos = spark.table("silver.dim_produtos")
df_categorias = spark.table("silver.dim_categoria_produtos_traducao")
df_avaliacoes = spark.table("silver.fat_avaliacoes_pedidos")
df_vendedores = spark.table("silver.dim_vendedores")
df_cotacao = spark.table("silver.dim_cotacao_dolar")

# 3. Preparando o Produto com a Categoria Traduzida em PT-BR
df_prod_cat = df_produtos.join(df_categorias, df_produtos.categoria_produto == df_categorias.nome_produto_en, "left") \
    .withColumn("categoria_pt", F.coalesce(F.col("nome_produto_pt"), F.col("categoria_produto"))) \
    .select("id_produto", "categoria_pt")

In [0]:
# 1. Cruzando Itens, Pedidos (para datas), Dólar e Produtos
df_base_vendas = df_itens.join(df_pedidos.select("id_pedido", "data_pedido"), "id_pedido", "inner") \
    .join(df_prod_cat, "id_produto", "left") \
    .join(df_cotacao, F.col("data_pedido") == F.col("data_cotacao"), "left")

# 2. Agregando as métricas de negócio
df_fat_vendas_comercial = df_base_vendas.groupBy(
    F.year("data_pedido").alias("ano_venda"),
    F.month("data_pedido").alias("mes_venda"),
    F.col("categoria_pt").alias("categoria_produto")
).agg(
    F.countDistinct("id_pedido").alias("total_pedidos"),
    F.count("id_item").alias("qtd_itens_vendidos"),
    # Soma financeira exata do preço dos itens
    F.round(F.sum("preco_BRL"), 2).alias("receita_total_brl"),
    F.round(F.sum(F.col("preco_BRL") / F.col("cotacao_venda")), 2).alias("receita_total_usd")
).withColumn(
    # Ticket Médio = Receita / Total de Pedidos
    "ticket_medio_brl", F.round(F.col("receita_total_brl") / F.col("total_pedidos"), 2)
)

# 3. Salvando na Camada Gold (usando overwriteSchema para tolerância a mudanças estruturais)
df_fat_vendas_comercial.write.format("delta") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("gold.fat_vendas_comercial")

print("Sucesso: gold.fat_vendas_comercial criada com os indicadores de receita!")

Sucesso: gold.fat_vendas_comercial criada com os indicadores de receita!


In [0]:
# Preparando a base de contagem por produto
df_ranking_produtos = df_base_vendas.groupBy("id_produto", F.col("categoria_pt").alias("categoria")) \
    .agg(F.count("id_item").alias("quantidade_vendida"))

# Top 5 MAIS Vendidos
print("--- TOP 5 PRODUTOS MAIS VENDIDOS ---")
display(df_ranking_produtos.orderBy(F.desc("quantidade_vendida")).limit(5))

# Top 5 MENOS Vendidos
print("--- TOP 5 PRODUTOS MENOS VENDIDOS ---")
display(df_ranking_produtos.orderBy(F.asc("quantidade_vendida")).limit(5))

--- TOP 5 PRODUTOS MAIS VENDIDOS ---


id_produto,categoria,quantidade_vendida
aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,527
99a4788cb24856965c36a24e339b6058,bed_bath_table,488
422879e10f46682990de24d770e7f83d,garden_tools,484
389d119b48cf3043d311335e499d9c6b,garden_tools,392
368c6c730842d78016ad823897a372db,garden_tools,388


--- TOP 5 PRODUTOS MENOS VENDIDOS ---


id_produto,categoria,quantidade_vendida
aacfae7cd4bac4849766f640abf2db8a,health_beauty,1
41402af2a88247152583bb812ba235dd,watches_gifts,1
1c0c0093a48f13ba70d0c6b0a9157cb7,furniture_decor,1
fa11ecd35f999783e96ac500532d9d45,cool_stuff,1
e9f2586707fb45ea0f9997c54f585842,sports_leisure,1


In [0]:
# 1. Cruzando Avaliações com Itens, Vendedores e Produtos
df_base_aval = df_avaliacoes.join(df_itens, "id_pedido", "inner") \
    .join(df_vendedores, "id_vendedor", "inner") \
    .join(df_prod_cat, "id_produto", "inner")

# 2. Construindo o Data Mart de Qualidade
df_fat_avaliacoes_clientes = df_base_aval.groupBy(
    F.col("categoria_pt").alias("categoria_produto"),
    F.col("id_vendedor").alias("nome_vendedor"),
    F.col("estado").alias("estado_vendedor")
).agg(
    F.count("id_avaliacao").alias("total_avaliacoes"),
    F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
    # Contagem condicional baseada na nota
    F.sum(F.when(F.col("nota_avaliacao") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
    F.sum(F.when(F.col("nota_avaliacao") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas")
).withColumn(
    # Cálculo do percentual de satisfação (Positivas / Total)
    "percentual_satisfacao",
    F.round((F.col("total_avaliacoes_positivas") / F.col("total_avaliacoes")) * 100, 2)
)

# 3. Salvando na Camada Gold
df_fat_avaliacoes_clientes.write.format("delta") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("gold.fat_avaliacoes_clientes")

print("Sucesso: gold.fat_avaliacoes_clientes criada com os indicadores de qualidade!")

Sucesso: gold.fat_avaliacoes_clientes criada com os indicadores de qualidade!


In [0]:
# --- Base de Qualidade de Produtos ---
df_qualidade_prod = df_base_aval.groupBy("id_produto", F.col("categoria_pt").alias("categoria")).agg(
    F.round(F.avg("nota_avaliacao"), 2).alias("nota_media"),
    F.count("id_avaliacao").alias("volume_avaliacoes")
)

print("1. O Produto MAIS bem avaliado:")
display(df_qualidade_prod.orderBy(F.desc("nota_media"), F.desc("volume_avaliacoes")).limit(1))

print("2. O Produto MENOS bem avaliado:")
display(df_qualidade_prod.orderBy(F.asc("nota_media"), F.desc("volume_avaliacoes")).limit(1))


# --- Base de Qualidade de Vendedores ---
df_qualidade_vend = df_base_aval.groupBy(F.col("id_vendedor").alias("nome_vendedor"), "estado").agg(
    F.round(F.avg("nota_avaliacao"), 2).alias("nota_media"),
    F.count("id_avaliacao").alias("volume_avaliacoes")
)

print("3. O Vendedor MAIS bem avaliado:")
display(df_qualidade_vend.orderBy(F.desc("nota_media"), F.desc("volume_avaliacoes")).limit(1))

print("4. O Vendedor MENOS bem avaliado:")
display(df_qualidade_vend.orderBy(F.asc("nota_media"), F.desc("volume_avaliacoes")).limit(1))

1. O Produto MAIS bem avaliado:


id_produto,categoria,nota_media,volume_avaliacoes
37eb69aca8718e843d897aa7b82f462d,garden_tools,5.0,15


2. O Produto MENOS bem avaliado:


id_produto,categoria,nota_media,volume_avaliacoes
05b515fdc76e888aada3c6d66c201dff,health_beauty,1.0,10


3. O Vendedor MAIS bem avaliado:


nome_vendedor,estado,nota_media,volume_avaliacoes
48efc9d94a9834137efd9ea76b065a38,PR,5.0,34


4. O Vendedor MENOS bem avaliado:


nome_vendedor,estado,nota_media,volume_avaliacoes
8d92f3ea807b89465643c219455e7369,SP,1.0,8


In [0]:
# Cruzando pedidos com vendedores para saber a origem e destino
df_logistica = spark.table("silver.fat_pedidos").select(
    "order_id", "status", "tempo_entrega_dias", "tempo_entrega_estimado_dias", "entrega_no_prazo"
)

df_insight_logistica = df_logistica.groupBy("entrega_no_prazo").agg(
    F.count("order_id").alias("total_pedidos"),
    F.round(F.avg("tempo_entrega_dias"), 1).alias("media_dias_entrega")
)

print("INSIGHT: Percentual de pedidos entregues no prazo vs atrasados")
display(df_insight_logistica)

INSIGHT: Percentual de pedidos entregues no prazo vs atrasados


entrega_no_prazo,total_pedidos,media_dias_entrega
Sim,88644,10.8
Não Entregue,2963,20.3
Não,7834,31.5


In [0]:
# Usando a fat_pagamentos que foi criada na Silver
df_pagamentos = spark.table("silver.fat_pagamentos_pedidos")

df_insight_pagamento = df_pagamentos.groupBy("tipo_pagamento").agg(
    F.count("id_pedido").alias("volume_pedidos"),
    F.round(F.sum("valor_pagamento"), 2).alias("receita_total_brl"),
    F.avg("parcelas_pagamento").alias("media_parcelas")
).orderBy(F.desc("receita_total_brl"))

print("INSIGHT: Faturamento por Método de Pagamento e média de parcelamento")
display(df_insight_pagamento)

INSIGHT: Faturamento por Método de Pagamento e média de parcelamento


tipo_pagamento,volume_pedidos,receita_total_brl,media_parcelas
Cartão de Crédito,76795,1.254208419E7,3.507155413763917
Boleto,19784,2869361.27,1.0
Voucher,5775,379436.87,1.0
Cartão de Débito,1529,217989.79,1.0
Não Definido,3,0.0,1.0


In [0]:
# Precisamos da tabela de clientes (customers) da Silver
df_clientes = spark.table("silver.dim_consumidores") # ajuste o nome se estiver diferente

df_geografia = df_clientes.groupBy("estado").agg(
    F.count("id_consumidor").alias("qtd_clientes"),
    F.countDistinct("cidade").alias("qtd_cidades_atendidas")
).orderBy(F.desc("qtd_clientes"))

print("INSIGHT: Top Estados com maior base de clientes")
display(df_geografia)

INSIGHT: Top Estados com maior base de clientes


estado,qtd_clientes,qtd_cidades_atendidas
SP,41746,629
RJ,12852,149
MG,11635,745
RS,5466,379
PR,5045,364
SC,3637,240
BA,3380,353
DF,2140,6
ES,2033,95
GO,2020,178
